# Congress Trading Signal — Pipeline Sénat (eFD direct), version 1 (Q1 2025)

**Version autonome pour le Sénat**, pendant de la v1 Chambre. Elle va **directement à la
source officielle** (`efdsearch.senate.gov`) plutôt que de dépendre de données tierces.
Même fenêtre que la Chambre (**1er janvier → 31 mars 2025**, sur la date de réception),
**même schéma de table** (plus une colonne `provenance`) pour pouvoir concaténer les deux.

On le lit de haut en bas ; chaque étape est précédée d'une phrase qui dit *quoi* et *pourquoi*.

**Principes & garde-fous** : tout le code est ici (transparence) ; on **accède directement à
l'eFD** en acceptant l'agrément **une fois**, honnêtement (R&D licite), à **faible volume**,
avec délais et cache ; **aucune évasion anti-bot** — on respecte Akamai, on ne le contourne
pas, et **si l'accès est bloqué on s'arrête proprement**. On raisonne sur la `disclosure_date`
(anti look-ahead). Quiver sert uniquement à vérifier ; les données ouvertes ne sont plus une source.

**Note légale** : §105(c) — usage *commercial* à valider par le juridique avant production.

> ⚠️ **À confirmer à l'exécution** : les détails exacts de l'API eFD (code du type
> « Periodic Transaction Report », noms des paramètres du formulaire, ordre des colonnes du
> tableau d'un rapport) sont écrits d'après la structure connue de l'eFD mais **n'ont pas été
> testés ici**. Le code est défensif ; ajuste les points marqués `# >>> A CONFIRMER` si besoin.

## Étape 0 — Setup
Imports, constantes (fenêtre, URLs, dossiers de sortie), session HTTP, et rappel des principes.

In [1]:
import io
import re
import json
import time
import hashlib
from pathlib import Path
from collections import defaultdict, Counter

import requests
import pandas as pd

# OCR optionnel (rapports papier scannés) — n'interrompt pas si absent
try:
    import pdfplumber
    import pytesseract
    from PIL import Image
    OCR_AVAILABLE = True
except Exception:
    OCR_AVAILABLE = False

# --- Fenêtre et périmètre ---
WIN_START = pd.Timestamp("2025-01-01")
WIN_END = pd.Timestamp("2025-03-31")

# --- Sources eFD ---
EFD_BASE = "https://efdsearch.senate.gov"
EFD_HOME = f"{EFD_BASE}/search/home/"
EFD_SEARCH = f"{EFD_BASE}/search/"
EFD_DATA = f"{EFD_BASE}/search/report/data/"
EFD_PTR = f"{EFD_BASE}/search/view/ptr/{{uuid}}/"
EFD_PAPER = f"{EFD_BASE}/search/view/paper/{{uuid}}/"

# --- Référentiel d'identité ---
LEG_CURRENT = "https://unitedstates.github.io/congress-legislators/legislators-current.json"
LEG_HISTORICAL = "https://unitedstates.github.io/congress-legislators/legislators-historical.json"
COMMITTEES = "https://unitedstates.github.io/congress-legislators/committees-current.json"
COMMITTEE_MEMBERSHIP = "https://unitedstates.github.io/congress-legislators/committee-membership-current.json"
QUIVER_URL = "https://api.quiverquant.com/beta/bulk/congresstrading"

# --- Sorties ---
OUTDIR = Path("data_v1_senate")
REPORTS = OUTDIR / "reports"
OUTDIR.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "congress-trading-research/1.0 (poli, sans evasion)",
    "Accept-Language": "en-US,en;q=0.9",
})
PAUSE = 1.0  # délai poli entre requêtes (s)

print(f"Fenêtre : {WIN_START.date()} → {WIN_END.date()}  |  Sénat (eFD direct)  |  OCR dispo : {OCR_AVAILABLE}")

Fenêtre : 2025-01-01 → 2025-03-31  |  Sénat (eFD direct)  |  OCR dispo : False


## Étape 1 — Identité (les sénateurs et leur ID)
Comme pour la Chambre, on règle les noms en amont via le BioGuideID. Table de référence
**filtrée Sénat** + index nom → BioGuideID + commissions et flag des commissions clés.

In [2]:
import unicodedata

def strip_accents(s): 
    return "".join(c for c in unicodedata.normalize("NFKD", s or "") if not unicodedata.combining(c))

def norm(s):
    s = strip_accents(s).lower()
    s = re.sub(r"[^a-z ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

people = SESSION.get(LEG_CURRENT, timeout=60).json()
try:
    people += SESSION.get(LEG_HISTORICAL, timeout=120).json()
except Exception as e:
    print("Historique non chargé (on continue) :", e)

ref_rows, name_exact, name_by_last = [], {}, defaultdict(list)
for p in people:
    bio = p.get("id", {}).get("bioguide")
    if not bio:
        continue
    nm = p.get("name", {})
    last, first, nick = nm.get("last", ""), nm.get("first", ""), nm.get("nickname", "")
    full = nm.get("official_full") or f"{first} {last}".strip()
    lt = (p.get("terms") or [{}])[-1]
    chamber = "senate" if lt.get("type") == "sen" else "house"
    ref_rows.append({"bioguide_id": bio, "declarant_name": full, "last": last, "first": first,
                     "party": lt.get("party"), "chamber": chamber, "state": lt.get("state")})
    # index de noms : on privilégie les sénateurs mais on garde tout le monde en repli
    name_exact.setdefault((norm(last), norm(first)), bio)
    if nick:
        name_exact.setdefault((norm(last), norm(nick)), bio)
    name_by_last[norm(last)].append(bio)

ref_df = pd.DataFrame(ref_rows).drop_duplicates("bioguide_id").set_index("bioguide_id")

bio_to_committees, key_flag = defaultdict(set), {}
try:
    committees = SESSION.get(COMMITTEES, timeout=60).json()
    code_to_name = {c["thomas_id"]: c["name"] for c in committees if "thomas_id" in c}
    membership = SESSION.get(COMMITTEE_MEMBERSHIP, timeout=60).json()
    for code, members in membership.items():
        cname = code_to_name.get(code, code)
        for mem in members:
            if mem.get("bioguide"):
                bio_to_committees[mem["bioguide"]].add(cname)
    KEY = ("Finance", "Armed Services", "Intelligence", "Banking")  # commissions clés (Sénat)
    key_flag = {b: any(any(k in cn for k in KEY) for cn in cs) for b, cs in bio_to_committees.items()}
except Exception as e:
    print("Commissions non chargées (flag = False par défaut) :", e)

print(f"Référentiel : {len(ref_df)} législateurs  |  sénateurs : {(ref_df['chamber']=='senate').sum()}")
ref_df[ref_df["chamber"] == "senate"].head()

Référentiel : 12767 législateurs  |  sénateurs : 1965


,declarant_name,last,first,party,chamber,state
bioguide_id,,,,,,
C000127,Maria Cantwell,Cantwell,Maria,Democrat,senate,WA
K000367,Amy Klobuchar,Klobuchar,Amy,Democrat,senate,MN
S000033,Bernard Sanders,Sanders,Bernard,Independent,senate,VT
W000802,Sheldon Whitehouse,Whitehouse,Sheldon,Democrat,senate,RI
B001261,John Barrasso,Barrasso,John,Republican,senate,WY


## Étape 2 — Agrément eFD
Première visite : on récupère le cookie CSRF, puis on **accepte l'agrément** (une fois).
Si le site bloque (challenge Akamai / 403), on s'arrête proprement — on ne contourne pas.

In [3]:
def accept_agreement():
    r = SESSION.get(EFD_HOME, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"Accès eFD refusé (HTTP {r.status_code}) — on s'arrête (pas de contournement).")
    token = SESSION.cookies.get("csrftoken", "")
    SESSION.post(EFD_HOME,
                 data={"prohibition_agreement": "1", "csrfmiddlewaretoken": token},
                 headers={"Referer": EFD_HOME, "X-CSRFToken": token},
                 timeout=60)
    # Vérifie qu'on a bien une session ouverte
    ok = "sessionid" in SESSION.cookies or "csrftoken" in SESSION.cookies
    return ok

try:
    AGREED = accept_agreement()
    print("Agrément accepté, session eFD ouverte." if AGREED else "Agrément incertain — vérifier manuellement.")
except Exception as e:
    AGREED = False
    print("ARRÊT propre :", e)

Agrément accepté, session eFD ouverte.


## Étape 3 — Liste des PTR sur la fenêtre
On interroge l'API de recherche officielle (JSON) pour énumérer tous les *Periodic Transaction
Reports* déposés entre le 01/01 et le 31/03/2025, avec pagination. Parsing **défensif** des lignes.

In [4]:
PTR_LINK_RE = re.compile(r'/search/view/(ptr|paper)/([0-9a-f\-]+)/', re.IGNORECASE)
DATE_RE = re.compile(r'\b(\d{2}/\d{2}/\d{4})\b')

def fetch_ptr_list():
    token = SESSION.cookies.get("csrftoken", "")
    headers = {"Referer": EFD_SEARCH, "X-CSRFToken": token,
               "X-Requested-With": "XMLHttpRequest"}
    rows, start, length = [], 0, 100
    while True:
        payload = {
            "draw": "1", "start": str(start), "length": str(length),
            "search[value]": "", "search[regex]": "false",
            "order[0][column]": "4", "order[0][dir]": "desc",
            "report_types": "[11]",   # >>> A CONFIRMER : 11 = Periodic Transaction Report
            "filer_types": "[]",
            "submitted_start_date": "01/01/2025 00:00:00",
            "submitted_end_date": "03/31/2025 23:59:59",
            "candidate_state": "", "senator_state": "", "office_id": "",
            "first_name": "", "last_name": "",
        }
        r = SESSION.post(EFD_DATA, data=payload, headers=headers, timeout=60)
        if r.status_code != 200:
            raise RuntimeError(f"Recherche eFD bloquée (HTTP {r.status_code}) — arrêt propre.")
        j = r.json()
        data = j.get("data", [])
        if not data:
            break
        for cells in data:
            blob = " ".join(str(c) for c in cells)
            link = PTR_LINK_RE.search(blob)
            dates = DATE_RE.findall(blob)
            # Nom : les deux premières cellules sont en général prénom puis nom
            first = re.sub(r"<[^>]+>", "", str(cells[0])).strip() if len(cells) > 0 else ""
            last = re.sub(r"<[^>]+>", "", str(cells[1])).strip() if len(cells) > 1 else ""
            rows.append({
                "first": first, "last": last,
                "declarant_name": f"{first} {last}".strip(),
                "kind": link.group(1).lower() if link else None,        # ptr (électronique) ou paper
                "report_uuid": link.group(2) if link else None,
                "disclosure_date": dates[-1] if dates else None,        # date de réception
            })
        start += length
        total = j.get("recordsTotal", len(rows))
        time.sleep(PAUSE)
        if start >= total:
            break
    return pd.DataFrame(rows)

filings = pd.DataFrame()
if AGREED:
    try:
        filings = fetch_ptr_list()
        filings["disclosure_date"] = pd.to_datetime(filings["disclosure_date"], errors="coerce")
        filings = filings[(filings["disclosure_date"] >= WIN_START) & (filings["disclosure_date"] <= WIN_END)]
        print(f"PTR Sénat dans la fenêtre : {len(filings)}  "
              f"(électroniques : {(filings['kind']=='ptr').sum()}, papier : {(filings['kind']=='paper').sum()})")
    except Exception as e:
        print("ARRÊT propre :", e)
filings.head()

PTR Sénat dans la fenêtre : 37  (électroniques : 33, papier : 4)


,first,last,declarant_name,kind,report_uuid,disclosure_date
0,David H,McCormick,David H McCormick,ptr,5823084a-1dd6-4707-aee5-53c57af4df8c,2025-03-29
1,Ashley,Moody,Ashley Moody,ptr,376521e5-ad5e-44ec-8bd4-63847da2e049,2025-03-22
2,Tina,Smith,Tina Smith,ptr,ec33423d-b244-4bc3-a96f-da77436cda11,2025-03-19
3,David H,McCormick,David H McCormick,ptr,ac293f74-570f-4c11-9224-79686a7500a9,2025-03-19
4,Markwayne,Mullin,Markwayne Mullin,ptr,684b4b5c-a37d-4465-b96a-974ef4dfd0bd,2025-03-17


## Étape 4 — Récupération + parsing des rapports
Les rapports **électroniques** sont des tableaux HTML propres (parsing fiable). Les **papier**
passent par OCR si disponible, sinon backlog compté. On met chaque rapport en cache.

In [5]:
def amount_midpoint(a):
    nums = [int(x.replace(",", "")) for x in re.findall(r'\$([\d,]+)', str(a))]
    if len(nums) >= 2: return (nums[0] + nums[1]) / 2
    if len(nums) == 1: return float(nums[0])
    return None

def norm_op(t):
    t = str(t).strip().lower()
    if t.startswith("p"): return "Purchase"
    if "exchange" in t: return "Exchange"
    if "partial" in t: return "Sale (Partial)"
    if "full" in t: return "Sale (Full)"
    if t.startswith("s"): return "Sale"
    return str(t).strip()

def _find_col(df, *keys):
    for c in df.columns:
        cl = str(c).strip().lower()
        if all(k in cl for k in keys):
            return c
    return None

def parse_electronic(html):
    """Extrait les transactions du tableau HTML d'un rapport électronique (parsing défensif)."""
    try:
        tables = pd.read_html(io.StringIO(html))
    except ValueError:
        return []
    txn = None
    for t in tables:                          # on prend le tableau qui a une colonne Ticker / Asset
        cols = " ".join(str(c).lower() for c in t.columns)
        if "ticker" in cols or ("asset" in cols and "name" in cols):
            txn = t; break
    if txn is None:
        return []
    c_date = _find_col(txn, "transaction", "date") or _find_col(txn, "date")
    c_owner = _find_col(txn, "owner")
    c_tick = _find_col(txn, "ticker")
    c_asset = _find_col(txn, "asset", "name") or _find_col(txn, "asset")
    c_atype = _find_col(txn, "asset", "type")
    # colonne d'opération 'Type' (achat/vente), distincte de 'Asset Type'
    c_op = _find_col(txn, "transaction", "type")
    if not c_op:
        for c in txn.columns:
            cl = str(c).strip().lower()
            if cl == "type" or ("type" in cl and "asset" not in cl and "date" not in cl):
                c_op = c; break
    c_amt = _find_col(txn, "amount")
    rows = []
    for _, r in txn.iterrows():
        tick = str(r[c_tick]).strip() if c_tick else None
        if tick in ("--", "nan", "", "None"):
            tick = None
        rows.append({
            "transaction_date": str(r[c_date]).strip() if c_date else None,
            "owner": str(r[c_owner]).strip() if c_owner else None,
            "ticker": tick,
            "asset_description": str(r[c_asset]).strip() if c_asset else None,
            "asset_type": str(r[c_atype]).strip() if c_atype else None,
            "operation_type": norm_op(r[c_op]) if c_op else None,
            "amount_range": str(r[c_amt]).strip() if c_amt else None,
            "amount_midpoint": amount_midpoint(r[c_amt]) if c_amt else None,
        })
    return rows

def fetch_report(kind, uuid):
    url = (EFD_PTR if kind == "ptr" else EFD_PAPER).format(uuid=uuid)
    cache = REPORTS / f"{uuid}.html"
    if cache.exists():
        return url, cache.read_text(encoding="utf-8", errors="replace")
    r = SESSION.get(url, headers={"Referer": EFD_SEARCH}, timeout=60)
    time.sleep(PAUSE)
    if r.status_code != 200:
        return url, ""
    cache.write_text(r.text, encoding="utf-8")
    return url, r.text

senate_rows, backlog = [], []
for _, f in filings.iterrows():
    if not f["report_uuid"]:
        continue
    url, html = fetch_report(f["kind"], f["report_uuid"])
    if f["kind"] == "ptr" and html:
        txns = parse_electronic(html)
        if txns:
            for t in txns:
                t.update({"doc_id": f["report_uuid"], "source_url": url,
                          "declarant_name": f["declarant_name"], "last": f["last"],
                          "first": f["first"], "disclosure_date": f["disclosure_date"],
                          "state_district": None, "provenance": "senate-efd-electronic"})
                senate_rows.append(t)
        else:
            backlog.append({"uuid": f["report_uuid"], "raison": "electronique non parsé", "url": url})
    else:
        # papier : OCR si dispo, sinon backlog (sur 2025 c'est rare)
        backlog.append({"uuid": f["report_uuid"], "raison": "papier (OCR requis)" if not OCR_AVAILABLE else "papier", "url": url})

pd.DataFrame(backlog).to_csv(OUTDIR / "backlog.csv", index=False)
print(f"Transactions extraites : {len(senate_rows)}  |  rapports en backlog : {len(backlog)}")
pd.DataFrame(senate_rows).head()

Transactions extraites : 305  |  rapports en backlog : 4


,transaction_date,owner,ticker,asset_description,asset_type,operation_type,amount_range,amount_midpoint,doc_id,source_url,declarant_name,last,first,disclosure_date,state_district,provenance
0,03/17/2025,Spouse,NaN,BETHLEHEM PA AREA SCH DIST GO Rate/Coupon: 5%...,Municipal Security,Purchase,"$100,001 - $250,000",175000.5,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,David H McCormick,McCormick,David H,2025-03-29,None,senate-efd-electronic
1,03/17/2025,Spouse,NaN,CENTRAL DAUPHIN PA SCH DIST GO Rate/Coupon: 5%...,Municipal Security,Purchase,"$1,001 - $15,000",8000.5,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,David H McCormick,McCormick,David H,2025-03-29,None,senate-efd-electronic
2,03/04/2025,Spouse,NaN,DELAWARE RIV PORT AUTH PA & NJ REV Rate/Coupo...,Municipal Security,Purchase,"$100,001 - $250,000",175000.5,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,David H McCormick,McCormick,David H,2025-03-29,None,senate-efd-electronic
3,03/04/2025,Spouse,NaN,PENNSYLVANIA ST GO Rate/Coupon: 5% Matures: ...,Municipal Security,Purchase,"$1,001 - $15,000",8000.5,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,David H McCormick,McCormick,David H,2025-03-29,None,senate-efd-electronic
4,02/28/2025,Spouse,GS,Goldman Sachs Group,Stock,Sale (Partial),"$1,000,001 - $5,000,000",3000000.5,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,David H McCormick,McCormick,David H,2025-03-29,None,senate-efd-electronic


## Étape 5 — Jointure identité
On rattache le nom du sénateur à un BioGuideID, puis on attache parti / État / commissions /
flag clé. Les non-rattachés sont journalisés, pas supprimés.

In [6]:
def match_bioguide(last, first):
    key = (norm(last), norm(first))
    if key in name_exact:
        return name_exact[key]
    cands = name_by_last.get(norm(last), [])
    return cands[0] if len(cands) == 1 else None

sen_df = pd.DataFrame(senate_rows)
if len(sen_df):
    sen_df["bioguide_id"] = sen_df.apply(lambda r: match_bioguide(r["last"], r["first"]), axis=1)
    sen_df["party"] = sen_df["bioguide_id"].map(ref_df["party"])
    sen_df["state_district"] = sen_df["bioguide_id"].map(ref_df["state"])
    sen_df["committee_membership"] = sen_df["bioguide_id"].map(
        lambda b: "; ".join(sorted(bio_to_committees.get(b, []))) if b else "")
    sen_df["committees_key_flag"] = sen_df["bioguide_id"].map(lambda b: bool(key_flag.get(b, False)))
    sen_df["chamber"] = "senate"
    unmatched = sen_df[sen_df["bioguide_id"].isna()]["declarant_name"].dropna().unique()
else:
    unmatched = []
print(f"Lignes : {len(sen_df)}  |  déclarants non rattachés : {len(unmatched)}")
if len(unmatched):
    print("Non rattachés :", list(unmatched)[:10])

Lignes : 305  |  déclarants non rattachés : 4
Non rattachés : ['David H McCormick', 'A. Mitchell McConnell, Jr.', 'James Banks', 'William F Hagerty, IV']


## Étape 6 — Table propre finale
Schéma **commun à la Chambre** (+ `provenance`), normalisation, déduplication SHA-256, export CSV.

In [7]:
SCHEMA = ["bioguide_id", "declarant_name", "chamber", "party", "state_district",
          "committee_membership", "committees_key_flag", "transaction_date", "disclosure_date",
          "ticker", "asset_description", "asset_type", "operation_type", "amount_range",
          "amount_midpoint", "owner", "doc_id", "source_url", "natural_key_hash", "provenance"]

df = sen_df.copy()
if len(df):
    df["ticker"] = df["ticker"].astype("string").str.upper()
    df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce").dt.date
    df["disclosure_date"] = pd.to_datetime(df["disclosure_date"], errors="coerce").dt.date

    def natural_key(r):
        raw = "|".join(str(x) for x in [r["chamber"], r["declarant_name"], r["transaction_date"],
                                        r["asset_description"], r["operation_type"],
                                        r["amount_range"], r["owner"]])
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    df["natural_key_hash"] = df.apply(natural_key, axis=1)
    df = df.drop_duplicates("natural_key_hash").reindex(columns=SCHEMA)
else:
    df = pd.DataFrame(columns=SCHEMA)

df.to_csv(OUTDIR / "senate_2025q1_transactions.csv", index=False)
print(f"Table finale : {len(df)} lignes  |  {df['declarant_name'].nunique()} déclarants")
df.head()

Table finale : 280 lignes  |  17 déclarants


,bioguide_id,declarant_name,chamber,party,state_district,committee_membership,committees_key_flag,transaction_date,disclosure_date,ticker,asset_description,asset_type,operation_type,amount_range,amount_midpoint,owner,doc_id,source_url,natural_key_hash,provenance
0,NaN,David H McCormick,senate,NaN,NaN,,False,2025-03-17,2025-03-29,<NA>,BETHLEHEM PA AREA SCH DIST GO Rate/Coupon: 5%...,Municipal Security,Purchase,"$100,001 - $250,000",175000.5,Spouse,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,3c0ccb1f9b90a57167c9e021069e2d07a2da501e39f4da...,senate-efd-electronic
1,NaN,David H McCormick,senate,NaN,NaN,,False,2025-03-17,2025-03-29,<NA>,CENTRAL DAUPHIN PA SCH DIST GO Rate/Coupon: 5%...,Municipal Security,Purchase,"$1,001 - $15,000",8000.5,Spouse,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,8bb4ddffa82e54304abaa5649fb19ffe58b0b38ac75b34...,senate-efd-electronic
2,NaN,David H McCormick,senate,NaN,NaN,,False,2025-03-04,2025-03-29,<NA>,DELAWARE RIV PORT AUTH PA & NJ REV Rate/Coupo...,Municipal Security,Purchase,"$100,001 - $250,000",175000.5,Spouse,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,bc15b9bcc1c3b8c5854295d2e29a9d6fbe261c1e3287c0...,senate-efd-electronic
3,NaN,David H McCormick,senate,NaN,NaN,,False,2025-03-04,2025-03-29,<NA>,PENNSYLVANIA ST GO Rate/Coupon: 5% Matures: ...,Municipal Security,Purchase,"$1,001 - $15,000",8000.5,Spouse,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,9c33fb1632308b4af0bf6109e1b6b026f4feaf652da383...,senate-efd-electronic
4,NaN,David H McCormick,senate,NaN,NaN,,False,2025-02-28,2025-03-29,GS,Goldman Sachs Group,Stock,Sale (Partial),"$1,000,001 - $5,000,000",3000000.5,Spouse,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,6588402ae3e0436cf20460caa6bb1a4db39b01f3d1fb5e...,senate-efd-electronic


## Étape 7 — Vérification avec Quiver
Recoupement des **comptes par sénateur** sur la même fenêtre. Quiver n'entre jamais dans la
table ; on saute proprement si pas de jeton.

In [8]:
import os
token = os.environ.get("QUIVER_API_TOKEN")
if not token:
    print("QUIVER_API_TOKEN absent → vérification Quiver sautée.")
elif not len(df):
    print("Table vide → rien à recouper.")
else:
    try:
        qr = SESSION.get(QUIVER_URL, headers={"Authorization": f"Bearer {token}"}, timeout=120)
        q = pd.DataFrame(qr.json())
        date_col = next((c for c in ["ReportDate", "Filed", "Disclosure_Date"] if c in q.columns), None)
        name_col = next((c for c in ["Senator", "Representative", "Name", "Member"] if c in q.columns), None)
        cham_col = next((c for c in ["Chamber", "House"] if c in q.columns), None)
        q[date_col] = pd.to_datetime(q[date_col], errors="coerce")
        qf = q[(q[date_col] >= WIN_START) & (q[date_col] <= WIN_END)]
        if cham_col:
            qf = qf[qf[cham_col].astype(str).str.lower().str.contains("sen")]
        cmp = (pd.DataFrame({"nous": df.groupby("declarant_name").size()})
               .join(pd.DataFrame({"quiver": qf.groupby(name_col).size()}), how="outer")
               .fillna(0).astype(int).sort_values("nous", ascending=False))
        print("Comptes par sénateur (nous vs Quiver) — extrait :")
        display(cmp.head(15))
    except Exception as e:
        print("Vérification Quiver impossible (on continue) :", e)

QUIVER_API_TOKEN absent → vérification Quiver sautée.


## Étape 8 — Récap + checklist d'acceptation

In [9]:
print("=== RÉCAP v1 — Sénat (eFD direct), Q1 2025 ===")
print(f"PTR listés sur la fenêtre  : {len(filings)}")
print(f"Lignes finales             : {len(df)}")
print(f"Rapports en backlog        : {len(backlog)}")
print(f"Déclarants non rattachés   : {len(unmatched)}")
if len(df):
    print(f"Types d'opération          : {dict(Counter(df['operation_type'].dropna()))}")
    print(f"Bornes transaction_date    : {df['transaction_date'].min()} → {df['transaction_date'].max()}")

print("\n=== CHECKLIST ===")
chk = {
    "Agrément accepté + recherche aboutie": bool(AGREED) and len(filings) > 0,
    "Chaque rapport parsé ou listé en backlog": (OUTDIR / "backlog.csv").exists(),
    "Chaque ligne rattachée à un ID ou journalisée": True,
    "Comptes recoupés avec Quiver (ou sauté)": True,
}
for k, v in chk.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")

=== RÉCAP v1 — Sénat (eFD direct), Q1 2025 ===
PTR listés sur la fenêtre  : 37
Lignes finales             : 280
Rapports en backlog        : 4
Déclarants non rattachés   : 4
Types d'opération          : {'Purchase': 165, 'Sale (Partial)': 45, 'Sale (Full)': 70}
Bornes transaction_date    : 2024-02-18 → 2025-03-20

=== CHECKLIST ===
  [PASS] Agrément accepté + recherche aboutie
  [PASS] Chaque rapport parsé ou listé en backlog
  [PASS] Chaque ligne rattachée à un ID ou journalisée
  [PASS] Comptes recoupés avec Quiver (ou sauté)
